In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import make_regression
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split

from deepordinal.torch import OrdinalOutput, ordinal_loss

In [ ]:
x, y = make_regression(n_samples=10000, n_features=8, n_informative=6, random_state=0)
y = QuantileTransformer().fit_transform(y.reshape(-1, 1))[:, 0]
y = np.clip(np.floor(y * 4), 0, 3).astype(np.int32)  # 4 ordinal classes: 0, 1, 2, 3

x_train, x_val, y_train, y_val = train_test_split(
    x.astype(np.float32), y, test_size=0.25, random_state=0
)
print("Class distribution:", np.bincount(y_train))

In [ ]:
torch.manual_seed(0)
features = nn.Sequential(nn.Linear(8, 16), nn.ReLU())
head = OrdinalOutput(input_dim=16, output_dim=4)
model = nn.Sequential(features, head)  # model(x) returns class probabilities
optimizer = torch.optim.Adam(model.parameters())
print(model)

In [ ]:
def loss_fn(xb, yb):
    return ordinal_loss(head.logit(features(xb)), yb, head.interior_thresholds)


train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)), batch_size=16)

epochs = 50
train_losses, val_losses = [], []
for epoch in range(epochs):
    epoch_loss = []
    for xb, yb in train_loader:
        loss = loss_fn(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
    train_losses.append(np.mean(epoch_loss))

    with torch.no_grad():
        val_losses.append(np.mean([loss_fn(xb, yb).item() for xb, yb in val_loader]))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}  train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

In [ ]:
with torch.no_grad():
    preds = model(torch.from_numpy(x_val)).argmax(dim=1).numpy()
print(f"Validation accuracy: {np.mean(preds == y_val):.4f}")

In [ ]:
plt.plot(train_losses, "--", label="train_loss")
plt.plot(val_losses, label="val_loss")
plt.title("Training and validation loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()